In [2]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

# ============================================================
# EXPERIMENT 10 – PIPE FLOW WITH TURBULENCE MODELS
# ROLL NUMBER: 20
# ============================================================

R = 0.025
NU = 1.5e-5
RHO = 1.2
U_TAU = 0.18
KAPPA = 0.41
A_PLUS = 26.0

BASE_N = 120
ROLL_NO = 20
N = BASE_N + ROLL_NO

MAX_ITER = 1000
TOL = 1.0e-10

BLASIUS_CONSTANT = 0.3164


def solve_pipe(N, u_tau, a_plus, model_name):

    h = R / N

    r = (np.arange(N) + 0.5) * h
    rf = np.arange(N + 1) * h

    source = 2.0 * u_tau**2 / R

    volume = np.pi * (rf[1:]**2 - rf[:-1]**2)
    area = 2.0 * np.pi * rf

    def solve_linear(eff):

        conductance = area[1:-1] * eff[1:-1] / h

        diagonal = np.zeros(N)

        diagonal[:-1] += conductance
        diagonal[1:] += conductance

        wall_conductance = area[-1] * NU / (0.5 * h)

        diagonal[-1] += wall_conductance

        matrix = diags(
            [-conductance, diagonal, -conductance],
            [-1, 0, 1],
            format="csr"
        )

        return spsolve(matrix, source * volume)

    velocity = solve_linear(np.full(N + 1, NU))

    for count in range(1, MAX_ITER + 1):

        slope = np.zeros(N + 1)

        slope[1:-1] = (
            velocity[1:] - velocity[:-1]
        ) / h

        slope[-1] = -velocity[-1] / (0.5 * h)

        y = R - rf

        mixing = KAPPA * y

        if model_name == "van Driest":

            mixing *= (
                1.0 - np.exp(
                    -y * u_tau / (a_plus * NU)
                )
            )

        eff = NU + mixing**2 * np.abs(slope)

        eff[-1] = NU

        new = solve_linear(eff)

        diff = float(np.max(np.abs(new - velocity)))

        velocity = 0.25 * new + 0.75 * velocity

        if diff < TOL:

            velocity = new
            break

    if count == MAX_ITER:

        raise RuntimeError(
            model_name + " Picard solver did not converge"
        )

    slope = np.zeros(N + 1)

    slope[1:-1] = (
        velocity[1:] - velocity[:-1]
    ) / h

    slope[-1] = -velocity[-1] / (0.5 * h)

    mixing = KAPPA * (R - rf)

    if model_name == "van Driest":

        mixing *= (
            1.0 - np.exp(
                -(R - rf) * u_tau / (a_plus * NU)
            )
        )

    eff = NU + mixing**2 * np.abs(slope)

    eff[-1] = NU

    residual = np.max(
        np.abs(solve_linear(eff) - velocity)
    )

    bulk = float(
        np.sum(volume * velocity) / (np.pi * R**2)
    )

    reynolds = 2.0 * R * bulk / NU

    darcy_f = 8.0 * (u_tau / bulk)**2

    blasius_f = BLASIUS_CONSTANT * reynolds**(-0.25)

    deviation = (
        abs(darcy_f - blasius_f) / blasius_f
    ) * 100.0

    return {
        "N": N,
        "velocity": velocity,
        "r": r,
        "rf": rf,
        "bulk": bulk,
        "reynolds": reynolds,
        "darcy_f": darcy_f,
        "blasius_f": blasius_f,
        "deviation": deviation,
        "iterations": count,
        "residual": residual,
        "u_tau": u_tau,
        "a_plus": a_plus
    }


# ============================================================
# MAIN SIMULATION – ROLL NUMBER 20
# N = 120 + 20 = 140
# ============================================================

main_results = {}

for model_name in ["Prandtl", "van Driest"]:

    main_results[model_name] = solve_pipe(
        N=N,
        u_tau=U_TAU,
        a_plus=A_PLUS,
        model_name=model_name
    )

print("\nMAIN SIMULATION – ROLL NUMBER 20")

print(f"Grid cells = {N}")
print(f"Radius = {R} m")
print(f"Kinematic viscosity = {NU} m2/s")
print(f"Friction velocity = {U_TAU} m/s")
print(f"A+ = {A_PLUS}")

for name, result in main_results.items():

    print(f"\n{name}")

    print(f"Bulk velocity = {result['bulk']:.6f} m/s")
    print(f"Re_D = {result['reynolds']:.2f}")
    print(f"Darcy friction factor = {result['darcy_f']:.6f}")
    print(f"Blasius friction factor = {result['blasius_f']:.6f}")
    print(f"Percentage deviation = {result['deviation']:.3f}%")
    print(f"Picard iterations = {result['iterations']}")
    print(f"Fixed-point error = {result['residual']:.3e}")


# ============================================================
# QUESTION 1 – GRID SENSITIVITY
# 60, 120 AND 240 RADIAL CELLS
# ============================================================

grid_values = [60, 120, 240]

grid_results = {
    "Prandtl": [],
    "van Driest": []
}

for model_name in ["Prandtl", "van Driest"]:

    for grid in grid_values:

        result = solve_pipe(
            N=grid,
            u_tau=U_TAU,
            a_plus=A_PLUS,
            model_name=model_name
        )

        grid_results[model_name].append(result)

print("\nQUESTION 1 – GRID SENSITIVITY")

print(
    "\nModel\t\tN\tBulk velocity\tDarcy friction"
)

for model_name in ["Prandtl", "van Driest"]:

    for result in grid_results[model_name]:

        print(
            f'{model_name:<16}'
            f'{result["N"]:<8}'
            f'{result["bulk"]:.6f}\t'
            f'{result["darcy_f"]:.6f}'
        )

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for model_name in ["Prandtl", "van Driest"]:

    axes[0].plot(
        [r["N"] for r in grid_results[model_name]],
        [r["bulk"] for r in grid_results[model_name]],
        "o-",
        label=model_name
    )

    axes[1].plot(
        [r["N"] for r in grid_results[model_name]],
        [r["darcy_f"] for r in grid_results[model_name]],
        "o-",
        label=model_name
    )

axes[0].set(
    xlabel="Radial cells (N)",
    ylabel="Bulk velocity (m/s)",
    title="Grid sensitivity of bulk velocity"
)

axes[1].set(
    xlabel="Radial cells (N)",
    ylabel="Darcy friction factor",
    title="Grid sensitivity of friction factor"
)

for ax in axes:

    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig("experiment_10_question_1.png", dpi=160)
plt.show()


# ============================================================
# QUESTION 2 – EFFECT OF A+
# A+ = 20, 26 AND 32
# KEEP U_TAU FIXED
# ============================================================

a_plus_values = [20, 26, 32]

a_plus_results = {
    "Prandtl": [],
    "van Driest": []
}

for model_name in ["Prandtl", "van Driest"]:

    for a_value in a_plus_values:

        result = solve_pipe(
            N=N,
            u_tau=U_TAU,
            a_plus=a_value,
            model_name=model_name
        )

        a_plus_results[model_name].append(result)

print("\nQUESTION 2 – EFFECT OF A+")

print(
    "\nModel\t\tA+\tBulk velocity\tDarcy friction"
)

for model_name in ["Prandtl", "van Driest"]:

    for result in a_plus_results[model_name]:

        print(
            f'{model_name:<16}'
            f'{result["a_plus"]:<8}'
            f'{result["bulk"]:.6f}\t'
            f'{result["darcy_f"]:.6f}'
        )

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for model_name in ["Prandtl", "van Driest"]:

    axes[0].plot(
        [r["a_plus"] for r in a_plus_results[model_name]],
        [r["bulk"] for r in a_plus_results[model_name]],
        "o-",
        label=model_name
    )

    axes[1].plot(
        [r["a_plus"] for r in a_plus_results[model_name]],
        [r["darcy_f"] for r in a_plus_results[model_name]],
        "o-",
        label=model_name
    )

axes[0].set(
    xlabel="A+",
    ylabel="Bulk velocity (m/s)",
    title="Effect of A+ on bulk velocity"
)

axes[1].set(
    xlabel="A+",
    ylabel="Darcy friction factor",
    title="Effect of A+ on friction factor"
)

for ax in axes:

    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig("experiment_10_question_2.png", dpi=160)
plt.show()


# ============================================================
# QUESTION 3 – INCREASE U_TAU TO 0.25 m/s
# COMPARE BOTH MODELS WITH BLASIUS
# ============================================================

NEW_U_TAU = 0.25

high_utau_results = {}

for model_name in ["Prandtl", "van Driest"]:

    high_utau_results[model_name] = solve_pipe(
        N=N,
        u_tau=NEW_U_TAU,
        a_plus=A_PLUS,
        model_name=model_name
    )

print("\nQUESTION 3 – INCREASED FRICTION VELOCITY")

print(f"New friction velocity = {NEW_U_TAU} m/s")

print(
    "\nModel\t\tUbulk\t\tRe_D\t\tDarcy f\t\tBlasius f\tDeviation (%)"
)

for model_name, result in high_utau_results.items():

    print(
        f'{model_name:<16}'
        f'{result["bulk"]:.6f}\t'
        f'{result["reynolds"]:.2f}\t'
        f'{result["darcy_f"]:.6f}\t'
        f'{result["blasius_f"]:.6f}\t'
        f'{result["deviation"]:.3f}'
    )

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for model_name, result in high_utau_results.items():

    axes[0].plot(
        result["velocity"],
        result["r"] / R,
        label=model_name
    )

    axes[1].bar(
        model_name,
        result["darcy_f"],
        label=model_name
    )

axes[0].set(
    xlabel="Velocity (m/s)",
    ylabel="r / R",
    title="Velocity profiles at u_tau = 0.25 m/s"
)

axes[1].set(
    ylabel="Darcy friction factor",
    title="Friction factor at increased u_tau"
)

axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("experiment_10_question_3.png", dpi=160)
plt.show()


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\nROLL NUMBER 20 – FINAL SUMMARY")

print(f"Main grid size = {N}")
print(f"Main friction velocity = {U_TAU}")
print(f"Main A+ = {A_PLUS}")

for model_name, result in main_results.items():

    print(
        f'{model_name}: '
        f'Ubulk={result["bulk"]:.6f}, '
        f'Re_D={result["reynolds"]:.2f}, '
        f'Darcy_f={result["darcy_f"]:.6f}'
    )

print("\nALL STUDENT EXERCISES COMPLETED")


MAIN SIMULATION – ROLL NUMBER 51
Grid cells = 171
Radius = 0.025 m
Kinematic viscosity = 1.5e-05 m2/s
Friction velocity = 0.18 m/s
A+ = 26.0

Prandtl
Bulk velocity = 1.567685 m/s
Re_D = 5225.62
Darcy friction factor = 0.105467
Blasius friction factor = 0.037214
Percentage deviation = 183.410%
Picard iterations = 56
Fixed-point error = 2.264e-11

van Driest
Bulk velocity = 2.615654 m/s
Re_D = 8718.85
Darcy friction factor = 0.037886
Blasius friction factor = 0.032743
Percentage deviation = 15.705%
Picard iterations = 65
Fixed-point error = 6.110e-12

QUESTION 1 – GRID SENSITIVITY

Model		N	Bulk velocity	Darcy friction
Prandtl         60      1.627859	0.097814
Prandtl         120     1.572805	0.104782
Prandtl         240     1.566546	0.105621
van Driest      60      2.619451	0.037776
van Driest      120     2.616125	0.037872
van Driest      240     2.615427	0.037892

QUESTION 2 – EFFECT OF A+

Model		A+	Bulk velocity	Darcy friction
Prandtl         20      1.567685	0.105467
Prandtl      